In [1]:

import pandas as pd
import re
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "Singlish Dataset.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Libraries imported successfully!")
  

Libraries imported successfully!


In [2]:

df = pd.read_csv(RAW_PATH)

print("Original dataset shape:", df.shape)
display(df.head())
  

Original dataset shape: (7832, 2)


,Comments,Sentiment
0,me price dala thiyana widiha hoidai wadi wisth...,Pos
1,ekata nam matath aduna chuti nangiyo oya ayyat...,Neu
2,hoda wada sathanak,Pos
3,ela ela aiya api oyata hamadamath adare,Pos
4,hikvision hodai cctv walata owa nam melo sasar...,Neg


In [3]:

def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["Cleaned_Comments"] = df["Comments"].apply(normalize_text)

display(df[["Comments", "Cleaned_Comments"]].head(10))
  

,Comments,Cleaned_Comments
0,me price dala thiyana widiha hoidai wadi wisth...,me price dala thiyana widiha hoidai wadi wisth...
1,ekata nam matath aduna chuti nangiyo oya ayyat...,ekata nam matath aduna chuti nangiyo oya ayyat...
2,hoda wada sathanak,hoda wada sathanak
3,ela ela aiya api oyata hamadamath adare,ela ela aiya api oyata hamadamath adare
4,hikvision hodai cctv walata owa nam melo sasar...,hikvision hodai cctv walata owa nam melo sasar...
5,aka para wata karala neluwanam iwarai,aka para wata karala neluwanam iwarai
6,make inna kochla settaka maru,make inna kochla settaka maru
7,pissuma heduna ekata gindara wage lassanai,pissuma heduna ekata gindara wage lassanai
8,coach kasun patta athal denne,coach kasun patta athal denne
9,athatama loku asadaranayak kale,athatama loku asadaranayak kale


In [4]:

print("Before cleaning:", df.shape)

df = df.dropna(subset=["Comments", "Sentiment"]).copy()

df = df[df["Cleaned_Comments"] != ""]

df = df.drop_duplicates(
    subset=["Cleaned_Comments", "Sentiment"]
).copy()

print("After cleaning:", df.shape)
print("\nSentiment distribution:")
display(df["Sentiment"].value_counts())
  

Before cleaning: (7832, 3)
After cleaning: (7831, 3)

Sentiment distribution:


Sentiment
Pos    4262
Neg    1853
Neu    1716
Name: count, dtype: int64

In [5]:

cleaned_df = df[["Cleaned_Comments", "Sentiment"]].rename(
    columns={"Cleaned_Comments": "Comments"}
)

output_path = PROCESSED_DIR / "singlish_cleaned.csv"

cleaned_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Cleaned dataset saved:", output_path)
display(cleaned_df.head())
  

Cleaned dataset saved: c:\Users\chami\OneDrive\Desktop\Singlish_Sentiment_Analysis\data\processed\singlish_cleaned.csv


,Comments,Sentiment
0,me price dala thiyana widiha hoidai wadi wisth...,Pos
1,ekata nam matath aduna chuti nangiyo oya ayyat...,Neu
2,hoda wada sathanak,Pos
3,ela ela aiya api oyata hamadamath adare,Pos
4,hikvision hodai cctv walata owa nam melo sasar...,Neg


In [6]:

X = cleaned_df["Comments"]
y = cleaned_df["Sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
  

Training samples: 6264
Testing samples: 1567


In [7]:

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)
print("Vocabulary size:", len(tfidf.get_feature_names_out()))
  

Training TF-IDF shape: (6264, 5000)
Testing TF-IDF shape: (1567, 5000)
Vocabulary size: 5000


In [8]:

feature_names = tfidf.get_feature_names_out()

print("First 30 TF-IDF features:")
print(feature_names[:30])

# Show non-zero TF-IDF features for one training comment
sample_vector = X_train_tfidf[0]

sample_features = pd.DataFrame({
    "Feature": feature_names[sample_vector.indices],
    "TF-IDF Score": sample_vector.data
}).sort_values("TF-IDF Score", ascending=False)

print("\nSample comment:", X_train.iloc[0])
display(sample_features.head(15))
  

First 30 TF-IDF features:
['aa' 'aadare' 'aadare karana' 'aadarei' 'aadarshayak' 'aadarshayak god'
 'aagamak' 'aah' 'aanduwa' 'aapu' 'aasa' 'aasai' 'aasama' 'aasawen'
 'aasawen baluwa' 'aawa' 'aba' 'ac' 'accent' 'accident' 'acer' 'act' 'ad'
 'ada' 'ada inna' 'ada kale' 'ada nam' 'ada oya' 'adahas' 'adahas tikak']

Sample comment: aiye matath build ekak karaganna one


,Feature,TF-IDF Score
9,karaganna one,0.405868
7,matath build,0.395031
6,aiye matath,0.386175
8,build ekak,0.341671
1,matath,0.313123
4,karaganna,0.309652
2,build,0.273537
5,one,0.243593
0,aiye,0.224332
3,ekak,0.186912
